# Módulo 6: Big data con pyspark - Ejercicio de evaluación

In [2]:
# Importación de librerías necesarias
import pyspark
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, FloatType, StringType, IntegerType
from pyspark.sql.functions import col, sum
from pyspark.ml.feature import StringIndexer, Imputer, OneHotEncoder, VectorAssembler, MinMaxScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import requests


## 1. Carga de datos de diamonds desde CSV con schema:

In [3]:
# Iniciar Spark Session
spark = SparkSession.builder.appName('pipeline_diamonds').getOrCreate()

# Cargar dataset
dataset_url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv'
csv_path = 'diamonds.csv'
with open(csv_path, 'wb') as file:
    file.write(requests.get(dataset_url).content)

# Definir esquema
diamonds_schema = StructType([
    StructField('carat', FloatType(), True),
    StructField('cut', StringType(), True),
    StructField('color', StringType(), True),
    StructField('clarity', StringType(), True),
    StructField('depth', FloatType(), True),
    StructField('table', FloatType(), True),
    StructField('price', IntegerType(), True),
    StructField('x', FloatType(), True),
    StructField('y', FloatType(), True),
    StructField('z', FloatType(), True),
])

df = spark.read.csv(csv_path, header=True, schema=diamonds_schema)

## 2. Pipeline de Regresión para predecir 'price'

In [4]:
# Manejo de valores nulos
df_reg = df.dropna(subset=['price'])
df_reg = df_reg.withColumnRenamed('price', 'label')

numerical_cols = [field.name for field in df_reg.schema.fields if isinstance(field.dataType, FloatType) and field.name != 'label']
categorical_cols = [field.name for field in df_reg.schema.fields if isinstance(field.dataType, StringType)]

# StringIndexer para variables categóricas
indexers = [StringIndexer(inputCol=c, outputCol=c + '_indexed', handleInvalid='keep') for c in categorical_cols]

# Imputación de valores faltantes
imputer_num = Imputer(inputCols=numerical_cols, outputCols=[c + '_imputed' for c in numerical_cols], strategy='median')

# OneHotEncoding para variables categóricas
ohe = [OneHotEncoder(inputCol=c + '_indexed', outputCol=c + '_onehot') for c in categorical_cols]

# Escalado
assembler_num = VectorAssembler(inputCols=[c + '_imputed' for c in numerical_cols], outputCol='numeric_features')
scaler = MinMaxScaler(inputCol='numeric_features', outputCol='numeric_features_scaled')
assembler_all = VectorAssembler(inputCols=['numeric_features_scaled'] + [c + '_onehot' for c in categorical_cols], outputCol='features')

# Modelo de regresión
regressor = RandomForestRegressor(seed=42)

# Pipeline regresión
pipeline_reg = Pipeline(stages=indexers + [imputer_num] + ohe + [assembler_num, scaler, assembler_all, regressor])

df_train_reg, df_test_reg = df_reg.randomSplit([0.8, 0.2], seed=42)
pipeline_model_reg = pipeline_reg.fit(df_train_reg)
df_pred_reg = pipeline_model_reg.transform(df_test_reg)

# Evaluación regresión
evaluator_r2 = RegressionEvaluator(metricName='r2')
print('R2:', evaluator_r2.evaluate(df_pred_reg))

R2: 0.9126982044190959


## 3. Pipeline de Clasificación para predecir 'cut'

In [ ]:
# Manejo de valores nulos
df_clf = df.dropna(subset=['cut'])
# Crear un nuevo DataFrame para clasificación
df_clf = df.withColumnRenamed("cut", "label")

# Aplicar preprocesamiento y pipeline a df_clf


# Indexar la variable objetivo
label_indexer = StringIndexer(inputCol='label', outputCol='label_indexed')

# Pipeline clasificación
classifier = RandomForestClassifier(seed=42, labelCol='label_indexed')
pipeline_clf = Pipeline(stages=indexers + [imputer_num] + ohe + [assembler_num, scaler, assembler_all, label_indexer, classifier])

df_train_clf, df_test_clf = df_clf.randomSplit([0.8, 0.2], seed=42)
pipeline_model_clf = pipeline_clf.fit(df_train_clf)
df_pred_clf = pipeline_model_clf.transform(df_test_clf)

# Evaluación clasificación
evaluator_f1 = MulticlassClassificationEvaluator(labelCol='label_indexed', metricName='f1')
print('F1 Score:', evaluator_f1.evaluate(df_pred_clf))

Py4JJavaError: An error occurred while calling o778.fit.
: org.apache.spark.SparkException: Input column cut does not exist.
	at org.apache.spark.ml.feature.StringIndexerBase.$anonfun$validateAndTransformSchema$2(StringIndexer.scala:128)
	at scala.collection.TraversableLike.$anonfun$flatMap$1(TraversableLike.scala:293)
	at scala.collection.IndexedSeqOptimized.foreach(IndexedSeqOptimized.scala:36)
	at scala.collection.IndexedSeqOptimized.foreach$(IndexedSeqOptimized.scala:33)
	at scala.collection.mutable.ArrayOps$ofRef.foreach(ArrayOps.scala:198)
	at scala.collection.TraversableLike.flatMap(TraversableLike.scala:293)
	at scala.collection.TraversableLike.flatMap$(TraversableLike.scala:290)
	at scala.collection.mutable.ArrayOps$ofRef.flatMap(ArrayOps.scala:198)
	at org.apache.spark.ml.feature.StringIndexerBase.validateAndTransformSchema(StringIndexer.scala:123)
	at org.apache.spark.ml.feature.StringIndexerBase.validateAndTransformSchema$(StringIndexer.scala:115)
	at org.apache.spark.ml.fe

## 4. GridSearch con CrossValidation

In [6]:
paramGrid = ParamGridBuilder().addGrid(regressor.numTrees, [10, 20]).build()
crossval = CrossValidator(estimator=pipeline_reg, estimatorParamMaps=paramGrid, evaluator=evaluator_r2, numFolds=3, parallelism=4, seed=42)
cv_model = crossval.fit(df_train_reg)
df_pred_cv = cv_model.transform(df_test_reg)
print('Best R2:', evaluator_r2.evaluate(df_pred_cv))

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

### Experimental 1:

In [ ]:

# Importación de librerías necesarias

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, MinMaxScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# 1. Carga de datos de diamonds desde CSV con schema:

# Descargar y guardar el dataset localmente
dataset_url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv'
csv_path = 'diamonds.csv'
df_pandas = pd.read_csv(dataset_url)
df_pandas.to_csv(csv_path, index=False)  # Guardar localmente

# Iniciar sesión de Spark
spark = SparkSession.builder.appName("DiamondsPipeline").getOrCreate()

# Definir el esquema explícito
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

schema = StructType([
    StructField("carat", DoubleType(), True),
    StructField("cut", StringType(), True),
    StructField("color", StringType(), True),
    StructField("clarity", StringType(), True),
    StructField("depth", DoubleType(), True),
    StructField("table", DoubleType(), True),
    StructField("price", DoubleType(), True),
    StructField("x", DoubleType(), True),
    StructField("y", DoubleType(), True),
    StructField("z", DoubleType(), True)
])

# Cargar datos con esquema explícito
df = spark.read.csv(csv_path, header=True, schema=schema)

# 2. Pipeline de Regresión para predecir 'price'

imputer = Imputer(inputCols=["carat", "depth", "table", "x", "y", "z"], outputCols=["carat", "depth", "table", "x", "y", "z"])

# Indexar variables categóricas
indexers = [StringIndexer(inputCol=col, outputCol=col + "_index") for col in ["cut", "color", "clarity"]]

# One-Hot Encoding
encoders = [OneHotEncoder(inputCol=col + "_index", outputCol=col + "_ohe") for col in ["cut", "color", "clarity"]]

# Convertir `carat` en vector antes del escalado**
carat_assembler = VectorAssembler(inputCols=["carat"], outputCol="carat_vector")
scaler = MinMaxScaler(inputCol="carat_vector", outputCol="carat_scaled")

# VectorAssembler para todas las features
assembler = VectorAssembler(inputCols=["carat_scaled", "depth", "table", "x", "y", "z", "cut_ohe", "color_ohe", "clarity_ohe"], outputCol="features")

# Modelo de regresión
regressor = RandomForestRegressor(featuresCol="features", labelCol="price")

# Pipeline de regresión
reg_pipeline = Pipeline(stages=[imputer] + indexers + encoders + [carat_assembler, scaler, assembler, regressor])

# Entrenar modelo
reg_model = reg_pipeline.fit(df)
df_pred_reg = reg_model.transform(df)

# Evaluación regresión
evaluator_reg = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="r2")
r2 = evaluator_reg.evaluate(df_pred_reg)
print(f"Regresión R2: {r2}")

# 3. PIPELINE DE CLASIFICACIÓN SOBRE 'cut'
label_indexer = StringIndexer(inputCol="cut", outputCol="label")
assembler_clf = VectorAssembler(inputCols=["carat_scaled", "depth", "table", "x", "y", "z", "color_ohe", "clarity_ohe"], outputCol="features")
classifier = RandomForestClassifier(featuresCol="features", labelCol="label")
clf_pipeline = Pipeline(stages=[imputer, label_indexer] + indexers[1:] + encoders[1:] + [carat_assembler, scaler, assembler_clf, classifier])
clf_model = clf_pipeline.fit(df)
df_pred_clf = clf_model.transform(df)

# Evaluación clasificación
evaluator_clf = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator_clf.evaluate(df_pred_clf)
print(f"Clasificación Accuracy: {accuracy}")

# 4. VALIDACIÓN CRUZADA SOBRE REGRESIÓN
paramGrid = ParamGridBuilder().addGrid(regressor.numTrees, [10, 20]).build()
crossval = CrossValidator(estimator=reg_pipeline, estimatorParamMaps=paramGrid, evaluator=evaluator_reg, numFolds=3)
cv_model = crossval.fit(df)
cv_r2 = evaluator_reg.evaluate(cv_model.transform(df))
print(f"Cross-Validation R2: {cv_r2}")

Regresión R2: 0.9112976052607803
Clasificación Accuracy: 0.6895810159436411
Cross-Validation R2: 0.9112976052607803


### Experimental 2:

In [40]:
# 1. Preprocesamiento

# Imputación de valores faltantes para características numéricas
imputer = Imputer(inputCols=["carat", "depth", "table", "x", "y", "z"], outputCols=["carat", "depth", "table", "x", "y", "z"])

# Indexación de variables categóricas (sin 'cut')
indexers = [StringIndexer(inputCol=col, outputCol=col + "_index") for col in ["color", "clarity"]]

# One-Hot Encoding para variables categóricas (sin 'cut')
encoders = [OneHotEncoder(inputCol=col + "_index", outputCol=col + "_ohe") for col in ["color", "clarity"]]

# VectorAssembler para características numéricas y categóricas
assembler = VectorAssembler(inputCols=["carat", "depth", "table", "x", "y", "z", "color_ohe", "clarity_ohe"], outputCol="features")

# 2. Modelo de regresión
regressor = RandomForestRegressor(featuresCol="features", labelCol="price")

# Pipeline de regresión
reg_pipeline = Pipeline(stages=[imputer] + indexers + encoders + [assembler, regressor])

# Entrenamiento y evaluación del modelo de regresión
reg_model = reg_pipeline.fit(df)
df_pred_reg = reg_model.transform(df)
evaluator_reg = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="r2")
r2 = evaluator_reg.evaluate(df_pred_reg)
print(f"Regresión R2: {r2}")

# 3. Modelo de clasificación (para la variable 'cut')
df_clf = df.withColumnRenamed("cut", "label")  # Renombrar 'cut' a 'label' para la clasificación

# Indexar la variable objetivo 'label'
label_indexer = StringIndexer(inputCol="label", outputCol="label_indexed")

# Modelo de clasificación
classifier = RandomForestClassifier(featuresCol="features", labelCol="label_indexed")

# Pipeline de clasificación
clf_pipeline = Pipeline(stages=[imputer] + indexers + encoders + [assembler, label_indexer, classifier])


# Entrenamiento y evaluación del modelo de clasificación
df_train_clf, df_test_clf = df_clf.randomSplit([0.8, 0.2], seed=42)


print(df_train_clf.columns)


clf_model = clf_pipeline.fit(df_train_clf)
df_pred_clf = clf_model.transform(df_test_clf)
evaluator_clf = MulticlassClassificationEvaluator(labelCol="label_indexed", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator_clf.evaluate(df_pred_clf)
print(f"Clasificación Accuracy: {accuracy}")

# 4. GridSearch con CrossValidation sobre el pipeline de regresión
paramGrid = ParamGridBuilder().addGrid(regressor.numTrees, [10, 20]).build()
crossval = CrossValidator(estimator=reg_pipeline, estimatorParamMaps=paramGrid, evaluator=evaluator_reg, numFolds=3)
cv_model = crossval.fit(df)
cv_r2 = evaluator_reg.evaluate(cv_model.transform(df))
print(f"Cross-Validation R2: {cv_r2}")


Regresión R2: 0.9109237881671257
['carat', 'label', 'color', 'clarity', 'depth', 'table', 'price', 'x', 'y', 'z']
Clasificación Accuracy: 0.6850879616837063
Cross-Validation R2: 0.9109237881671257
